In [4]:
import tarfile
import pandas as pd
import io
import time
import sys

archive_path = r"D:\ECE 247\emg2qwerty_W26\data\rawdata.tar.gz"

def log(msg):
    print(msg, flush=True)

# Helps some environments flush line-by-line
try:
    sys.stdout.reconfigure(line_buffering=True)
except Exception:
    pass

t0 = time.perf_counter()
metadata = None
files_seen = 0
last_update = t0

log(f"Opening archive: {archive_path}")

with tarfile.open(archive_path, "r|gz") as tar:
    log("Archive opened. Beginning sequential scan...")

    for member in tar:
        files_seen += 1
        now = time.perf_counter()

        # Periodic live status update
        if now - last_update >= 1.0:
            log(
                f"[{now - t0:7.1f}s] Scanned {files_seen:,} entries "
                f"(current: {member.name})"
            )
            last_update = now

        if member.isfile() and member.name.lower().endswith("metadata.csv"):
            log(f"Found metadata candidate: {member.name}")
            log("Extracting file bytes...")

            with tar.extractfile(member) as f:
                raw = f.read()

            log(f"Read {len(raw):,} bytes. Parsing CSV...")
            metadata = pd.read_csv(io.BytesIO(raw))

            log(f"Loaded metadata from: {member.name}")
            log(f"Metadata shape: {metadata.shape}")
            log(f"Total elapsed: {time.perf_counter() - t0:.1f} s")
            break

if metadata is None:
    raise FileNotFoundError("metadata.csv not found inside archive")

log("Columns:")
log(str(list(metadata.columns)))

metadata.head()

Opening archive: D:\ECE 247\emg2qwerty_W26\data\rawdata.tar.gz
Archive opened. Beginning sequential scan...
[    1.4s] Scanned 3 entries (current: emg2qwerty-data-2021-08/2020-08-13-1597357485-keystrokes-71409769.hdf5)
[    2.5s] Scanned 4 entries (current: emg2qwerty-data-2021-08/2020-08-13-1597363426-keystrokes-71409769.hdf5)
[    3.7s] Scanned 5 entries (current: emg2qwerty-data-2021-08/2020-08-13-1597369024-keystrokes-71409769.hdf5)
[    4.9s] Scanned 6 entries (current: emg2qwerty-data-2021-08/2020-08-14-1597444340-keystrokes-71409769.hdf5)
[    6.3s] Scanned 7 entries (current: emg2qwerty-data-2021-08/2020-08-15-1597560433-keystrokes-71409769.hdf5)
[    7.7s] Scanned 8 entries (current: emg2qwerty-data-2021-08/2020-08-17-1597699312-keystrokes-09456349.hdf5)
[    9.4s] Scanned 10 entries (current: emg2qwerty-data-2021-08/2020-08-17-1597701568-keystrokes-09456349.hdf5)
[   11.8s] Scanned 12 entries (current: emg2qwerty-data-2021-08/2020-08-17-1597730688-keystrokes-71409769.hdf5)
[ 

,user,session,condition,duration_mins,num_keystrokes,num_prompts,quality_check_tags
0,71409769,2020-08-13-1597363426-keystrokes-71409769,on_keyboard,17.144215,4748,158,[]
1,71409769,2020-08-14-1597444340-keystrokes-71409769,on_keyboard,17.159030,4577,158,[]
2,9456349,2020-08-13-1597354281-keystrokes,on_keyboard,10.399901,4362,157,[]
3,9456349,2020-08-13-1597355141-keystrokes,on_keyboard,9.577550,4380,158,[]
4,71409769,2020-08-15-1597560433-keystrokes-71409769,on_keyboard,15.559813,4604,158,[]


In [5]:
metadata_cache_path = r"D:\ECE 247\emg2qwerty_W26\data\metadata_cached.csv"

metadata.to_csv(metadata_cache_path, index=False)
print(f"Saved cached metadata to: {metadata_cache_path}")

Saved cached metadata to: D:\ECE 247\emg2qwerty_W26\data\metadata_cached.csv


In [6]:
import os
csv_path = r"D:\ECE 247\emg2qwerty_W26\data\metadata_cached.csv"

if os.path.exists(csv_path):
    metadata = pd.read_csv(csv_path)
    print("Loaded metadata from CSV cache.")
else:
    raise FileNotFoundError("No cached metadata file found.")

print(metadata.shape)

Loaded metadata from CSV cache.
(1135, 7)


In [7]:
required_cols = ["user", "session"]
missing = [c for c in required_cols if c not in metadata.columns]
if missing:
    raise ValueError(f"Metadata is missing required columns: {missing}")

# Normalize types
metadata["user"] = metadata["user"].astype(str)
metadata["session"] = metadata["session"].astype(str)

# Build several lookup keys to make matching more robust
session_to_user = {}

for _, row in metadata[["user", "session"]].drop_duplicates().iterrows():
    user = row["user"]
    session = row["session"].replace("\\", "/")
    base = os.path.basename(session)

    candidates = {
        session,
        base,
        f"{session}.h5",
        f"{session}.hdf5",
        f"{base}.h5",
        f"{base}.hdf5",
    }

    for key in candidates:
        session_to_user[key] = user

print(f"Built lookup with {len(session_to_user):,} keys")

Built lookup with 3,405 keys


In [8]:
import tarfile
import shutil
import time
from pathlib import Path

archive_path = r"D:\ECE 247\emg2qwerty_W26\data\rawdata.tar.gz"
output_root = Path(r"D:\ECE 247\emg2qwerty_W26\data")

COPY_BUF = 16 * 1024 * 1024   # 16 MB
REPORT_EVERY = 20           # status update every N archive entries

start = time.perf_counter()
scanned = 0
matched = 0
written = 0
skipped_unmatched = 0

print(f"Opening archive: {archive_path}", flush=True)
print(f"Writing reorganized files under: {output_root}", flush=True)

with tarfile.open(archive_path, "r|gz") as tar:
    for member in tar:
        scanned += 1

        if scanned % REPORT_EVERY == 0:
            elapsed = time.perf_counter() - start
            rate = scanned / elapsed if elapsed > 0 else 0
            print(
                f"[{elapsed:8.1f}s] scanned={scanned:,} matched={matched:,} "
                f"written={written:,} unmatched={skipped_unmatched:,} "
                f"rate={rate:,.1f} entries/s",
                flush=True
            )

        if not member.isfile():
            continue

        member_name = member.name.replace("\\", "/")
        base_name = os.path.basename(member_name)

        if not base_name.lower().endswith((".h5", ".hdf5")):
            continue

        # Try a few matching strategies
        stem = os.path.splitext(base_name)[0]

        user = (
            session_to_user.get(member_name)
            or session_to_user.get(base_name)
            or session_to_user.get(stem)
        )

        if user is None:
            skipped_unmatched += 1
            if skipped_unmatched <= 10:
                print(f"Unmatched HDF5: {member_name}", flush=True)
            continue

        matched += 1

        user_dir = output_root / user
        user_dir.mkdir(parents=True, exist_ok=True)

        dest_path = user_dir / base_name

        print(
            f"[{time.perf_counter() - start:8.1f}s] "
            f"user={user}  file={base_name}",
            flush=True
        )

        src = tar.extractfile(member)
        if src is None:
            print(f"  Could not extract {member_name}", flush=True)
            continue

        with open(dest_path, "wb") as dst:
            shutil.copyfileobj(src, dst, length=COPY_BUF)

        written += 1

elapsed = time.perf_counter() - start
print("\nDone.", flush=True)
print(f"Elapsed: {elapsed:.1f} s", flush=True)
print(f"Scanned:   {scanned:,}", flush=True)
print(f"Matched:   {matched:,}", flush=True)
print(f"Written:   {written:,}", flush=True)
print(f"Unmatched: {skipped_unmatched:,}", flush=True)

Opening archive: D:\ECE 247\emg2qwerty_W26\data\rawdata.tar.gz
Writing reorganized files under: D:\ECE 247\emg2qwerty_W26\data
[     0.0s] user=9456349  file=2020-08-13-1597354281-keystrokes.hdf5
[     1.1s] user=9456349  file=2020-08-13-1597355141-keystrokes.hdf5
[     2.2s] user=71409769  file=2020-08-13-1597357485-keystrokes-71409769.hdf5
[     4.0s] user=71409769  file=2020-08-13-1597363426-keystrokes-71409769.hdf5
[     5.9s] user=71409769  file=2020-08-13-1597369024-keystrokes-71409769.hdf5
[     7.5s] user=71409769  file=2020-08-14-1597444340-keystrokes-71409769.hdf5
[     9.4s] user=71409769  file=2020-08-15-1597560433-keystrokes-71409769.hdf5
[    11.2s] user=9456349  file=2020-08-17-1597699312-keystrokes-09456349.hdf5
[    13.9s] user=9456349  file=2020-08-17-1597700134-keystrokes-09456349.hdf5
[    18.9s] user=9456349  file=2020-08-17-1597701568-keystrokes-09456349.hdf5
[    23.7s] user=71409769  file=2020-08-17-1597712595-keystrokes-71409769.hdf5
[    27.8s] user=71409769  

OSError: [Errno 28] No space left on device

In [7]:
import csv
import pandas as pd
from pathlib import Path

src = Path("./data/metadata_cached.csv")
dst = Path("./data/metadata_user_session.csv")

written = 0
skipped = 0

with open(src, "r", encoding="utf-8", errors="replace", newline="") as fin, \
     open(dst, "w", encoding="utf-8", newline="") as fout:

    writer = csv.writer(fout)
    writer.writerow(["user", "session"])

    header = fin.readline()  # skip original header

    for line_no, line in enumerate(fin, start=2):
        line = line.rstrip("\n")
        if not line:
            continue

        # Only take the first two comma-separated fields
        parts = line.split(",", 2)

        if len(parts) < 2:
            skipped += 1
            print(f"Skipping malformed line {line_no}: {line[:120]}")
            continue

        user = parts[0].strip().strip('"')
        session = parts[1].strip().strip('"')

        writer.writerow([user, session])
        written += 1

print(f"Created: {dst.resolve()}")
print(f"Rows written: {written}")
print(f"Rows skipped: {skipped}")

df = pd.read_csv(dst)
print(df.shape)
print(df.head())

Created: D:\ECE 247\emg2qwerty_W26\data\metadata_user_session.csv
Rows written: 1135
Rows skipped: 0
(1135, 2)
       user                                    session
0  71409769  2020-08-13-1597363426-keystrokes-71409769
1  71409769  2020-08-14-1597444340-keystrokes-71409769
2   9456349           2020-08-13-1597354281-keystrokes
3   9456349           2020-08-13-1597355141-keystrokes
4  71409769  2020-08-15-1597560433-keystrokes-71409769


In [10]:
from yaml_creation import make_dataset_yaml

users = ["89335547", "41556660", "14312238"]

yaml_text, dataset = make_dataset_yaml(
    users=users,
    data_root="./data/",
    output_path="./config/user/three_user_dataset.yaml",
    metadata_csv="./data/metadata_user_session.csv",
    train_frac=0.8,
    val_frac=0.1,
    test_frac=0.1,
    split_mode="random",
    seed=42,
    require_existing_files=True,
    user_label="multi_user"
)

print(yaml_text[:2000])

Loading metadata from csv: data\metadata_user_session.csv
Building YAML for users: ['89335547', '41556660', '14312238']
Fractions: train=0.8, val=0.1, test=0.1
Split mode: random

User 89335547
  Sessions in metadata: 18
  Sessions found in folder: 18
  Sessions kept after intersection: 18
  train=14, val=2, test=2

User 41556660
  Sessions in metadata: 12
  Sessions found in folder: 12
  Sessions kept after intersection: 12
  train=10, val=1, test=1

User 14312238
  Sessions in metadata: 11
  Sessions found in folder: 11
  Sessions kept after intersection: 11
  train=9, val=1, test=1

Saved YAML to: config\user\three_user_dataset.yaml
Total train: 33
Total val:   4
Total test:  4
# @package _global_
user: multi_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-03-1622766673-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    sess